# Testing with Marko
Why do we get problems on certain datasets with gran approx rules?

Update 11/04/24, before meeting with Marko:
- it should work without the reducts -> adapting this

In [41]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
import numpy as np
from quickrules.data_handling import get_dataset
from pathlib import Path
import fuzzyroughrules.fuzzy_operators as fo
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

In [43]:
name = 'ecoli' # 'bupa
nr = 6

In [44]:
dataset = pd.read_csv(Path.cwd() / 'keel-data' / f'{name}-10-fold' / f'{name}-10-6tra.dat', header=None, comment='@')
nums = [t != 'object' for t in dataset.dtypes]
nums[-1] = False
x_dataset = dataset.loc[:, nums]
x_train, y_train = x_dataset.values, dataset.iloc[:, -1].values

In [45]:
y_train

array([' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp',
       ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp',
       ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp',
       ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp',
       ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp',
       ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp',
       ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp',
       ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp',
       ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp',
       ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp',
       ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp',
       ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp',
       ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp',
       ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp', ' cp',
       ' cp', ' cp',

In [46]:
x_train, y_train = get_dataset(
    Path.cwd() / 'keel-data' / f'{name}-10-fold',
    f"{nr}tra"
)
x_test, y_test = get_dataset(
    Path.cwd() / 'keel-data' / f'{name}-10-fold',
    f"{nr}tst"
)

In [47]:
classes = list(np.unique(np.append(y_train, y_test)))
y_train = np.array([classes.index(label) for label in y_train])
y_test = np.array([classes.index(label) for label in y_test])

In [48]:
X = np.atleast_2d(x_train)
n_samples = X.shape[0]
n_attributes = X.shape[1]
n_classes = len(np.unique(y_train))
scaler = MinMaxScaler()
scaler.fit(X)
X_scaled = scaler.transform(X)
rel_matrix_x = fo.triangular_similarity(X_scaled, X_scaled)
rel_matrix_y = fo.discernibility_matrix(y_train, y_train)

In [49]:
n_classes

8

In [50]:
positive_region = fo.lukasiewicz_t_norm(
    fo.get_multiclass_granular_approx_mse_cvxopt(rel_matrix_x, rel_matrix_y),
    .5
)

In [51]:
positive_region

array([0.21254, 0.27519, 0.11505, 0.12588, 0.31600, 0.11607, 0.31963,
       0.33847, 0.10315, 0.27491, 0.24216, 0.27054, 0.27996, 0.14711,
       0.27170, 0.17842, 0.25663, 0.09240, 0.35046, 0.14268, 0.39286,
       0.36576, 0.13665, 0.40746, 0.23415, 0.17842, 0.29670, 0.30745,
       0.21398, 0.21410, 0.22777, 0.26735, 0.13246, 0.27519, 0.35396,
       0.18091, 0.22559, 0.10754, 0.26408, 0.21168, 0.22444, 0.21068,
       0.23218, 0.29670, 0.19992, 0.06014, 0.22673, 0.29670, 0.18762,
       0.39430, 0.08165, 0.14990, 0.29668, 0.16486, 0.13019, 0.24075,
       0.14616, 0.38731, 0.20266, 0.09240, 0.29670, 0.11256, 0.31434,
       0.21878, 0.34113, 0.18153, 0.26444, 0.24824, 0.19592, 0.15160,
       0.24217, 0.24293, 0.27312, 0.19315, 0.25600, 0.19117, 0.12465,
       0.15691, 0.15139, 0.32546, 0.17241, 0.21206, 0.35368, 0.19966,
       0.34696, 0.24509, 0.00000, 0.14704, 0.11676, 0.28347, 0.20693,
       0.26663, 0.21794, 0.35396, 0.30745, 0.14427, 0.16767, 0.15352,
       0.30153, 0.30

In [52]:
def get_reducts(X, with_reducts=True):
    reducts = []
    current_attributes = np.random.permutation(np.arange(n_attributes))
    for i in range(n_samples):
        decision_set = rel_matrix_y[i]
        selected_types = 2*np.ones(n_attributes, dtype=int)
        if with_reducts:
            for elem in current_attributes:
                tmp_types = selected_types
                tmp_types[elem] = 0
                new_granule = fo.lukasiewicz_t_norm(fo.general_triangular_relation(X, X[i], 1, tmp_types), positive_region[i])
                if fo.is_subset(new_granule, decision_set, 1e-4):
                    selected_types = tmp_types
                    continue
                tmp_types[elem] = 1
                new_granule = fo.lukasiewicz_t_norm(fo.general_triangular_relation(X, X[i], 1, tmp_types), positive_region[i])
                if fo.is_subset(new_granule, decision_set, 1e-4):
                    selected_types = tmp_types
                    continue
                tmp_types[elem] = -1
                new_granule = fo.lukasiewicz_t_norm(fo.general_triangular_relation(X, X[i], 1, tmp_types), positive_region[i])
                if fo.is_subset(new_granule, decision_set, 1e-4):
                    selected_types = tmp_types
                    continue
                tmp_types[elem] = 2
        reducts.append(selected_types)
    return np.array(reducts)

In [53]:
reducts = get_reducts(X_scaled,False)

In [54]:
covering = np.zeros((n_samples, n_samples))
for i in range(n_samples):
    current_covering = fo.lukasiewicz_t_norm(fo.general_triangular_relation(X_scaled, X_scaled[i], 1, reducts[i]), positive_region[i])
    if len(current_covering.shape) > 1:
        current_covering = current_covering.T
    # print(current_covering)
    covering[i, :] = current_covering

In [55]:
covering

array([[0.21254, 0.00000, 0.00000, ..., 0.00000, 0.00000, 0.00000],
       [0.00000, 0.27519, 0.00000, ..., 0.00000, 0.00000, 0.00000],
       [0.00000, 0.00000, 0.11505, ..., 0.00000, 0.03425, 0.00000],
       ...,
       [0.00000, 0.00000, 0.00000, ..., 0.00000, 0.00000, 0.00000],
       [0.00000, 0.00000, 0.00000, ..., 0.00000, 0.00000, 0.00000],
       [0.00000, 0.00000, 0.00000, ..., 0.00000, 0.00000, 0.12177]])

In [56]:
np.max(covering[:, 246])

9.776467857491866e-09

In [57]:
np.sort(-rel_matrix_x[246])

array([-1.00000, -0.88506, -0.82759, -0.78161, -0.77419, -0.76136,
       -0.75862, -0.74194, -0.74157, -0.73563, -0.73118, -0.72727,
       -0.72727, -0.72414, -0.71717, -0.70455, -0.70115, -0.69318,
       -0.69318, -0.68817, -0.68182, -0.67045, -0.66667, -0.66667,
       -0.66667, -0.65909, -0.65909, -0.65909, -0.65909, -0.64773,
       -0.64773, -0.64773, -0.64773, -0.64045, -0.64045, -0.63636,
       -0.63636, -0.63636, -0.63636, -0.63636, -0.63636, -0.62921,
       -0.61364, -0.61364, -0.61364, -0.61364, -0.61364, -0.61364,
       -0.61364, -0.61364, -0.60674, -0.60674, -0.60227, -0.60227,
       -0.60227, -0.60227, -0.60227, -0.59551, -0.59091, -0.59091,
       -0.59091, -0.59091, -0.59091, -0.58427, -0.57955, -0.57955,
       -0.57955, -0.57955, -0.57576, -0.57576, -0.57303, -0.56818,
       -0.56818, -0.56818, -0.56818, -0.56818, -0.56180, -0.56180,
       -0.56180, -0.56180, -0.55682, -0.55682, -0.55682, -0.55682,
       -0.55682, -0.55056, -0.55056, -0.54545, -0.54545, -0.54

In [58]:
np.argmin(np.max(covering.T, axis=0))

86

In [59]:
np.max(covering[86])

0.0

In [60]:
from fuzzyroughrules.mse_gran_approx_rules import RuleGenerator

In [61]:
model = RuleGenerator(with_reducts=False)

In [62]:
model.fit(x_train, y_train,_)

AttributeError: Unable to retrieve attribute 'x'

In [ ]:
model.selected_indexes

In [ ]:
y_pred = model.predict(x_test)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)

In [ ]:
y_test

In [ ]:
y_pred